In [2]:
import cv2
import numpy as np
from rtmlib import Body

COCO_CONNECTIONS = [
    (0, 1), (0, 2), (1, 3), (2, 4),           # head
    (5, 7), (7, 9), (6, 8), (8, 10),           # arms
    (5, 6), (5, 11), (6, 12), (11, 12),        # torso
    (11, 13), (13, 15), (12, 14), (14, 16),    # legs
]

# Per-joint confidence thresholds used for drawing.
# Swimming legs are often blurred/underwater, so low scores should not move the
# tracker even if they are sometimes good enough to keep a faded point visible.
_DEFAULT_THR = np.full(17, 0.35)
_DEFAULT_THR[7:11]  = 0.15   # elbows + wrists
_DEFAULT_THR[11:13] = 0.18   # hips
_DEFAULT_THR[13:15] = 0.25   # knees
_DEFAULT_THR[15:17] = 0.30   # ankles

# Stricter thresholds for updating the temporal filter state. This prevents
# one bad low-confidence kick frame from pulling the leg skeleton away.
_UPDATE_THR = _DEFAULT_THR.copy()
_UPDATE_THR[11:13] = 0.15
_UPDATE_THR[13:15] = 0.22
_UPDATE_THR[15:17] = 0.28

# Per-joint cap on one-frame motion, in units of the body (torso) scale.
# Crawl recovery whips the wrist fast, but a *whole* skeleton collapsing onto
# the body center is the main failure mode when an arm passes above the head:
# RTMPose tends to emit confidently-wrong limb positions near the crop center.
# These limits gate each joint independently so collapses are filtered out
# while real swimming motion still passes through.
_MAX_JUMP_BY_JOINT = np.full(17, 1.10, dtype=np.float64)
_MAX_JUMP_BY_JOINT[0:5]   = 0.45   # head / face — moves slowly
_MAX_JUMP_BY_JOINT[5:7]   = 0.45   # shoulders   — anchored to torso
_MAX_JUMP_BY_JOINT[7:9]   = 1.00   # elbows
_MAX_JUMP_BY_JOINT[9:11]  = 1.40   # wrists      — fastest during recovery
_MAX_JUMP_BY_JOINT[11:13] = 0.45   # hips        — anchored to torso
_MAX_JUMP_BY_JOINT[13:15] = 1.10   # knees
_MAX_JUMP_BY_JOINT[15:17] = 1.30   # ankles      — kick flick

# RTMPose-L body7, 384x288 — much more accurate than YOLO-pose / MediaPipe on
# unusual body poses (swimming, gymnastics, etc.). Uses YOLOX-M as detector.
_RTMPOSE_L_URL = (
    "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/"
    "rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.zip"
)
_YOLOX_M_URL = (
    "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/"
    "yolox_m_8xb8-300e_humanart-c2c7a14a.zip"
)

_model = Body(
    det=_YOLOX_M_URL,
    det_input_size=(640, 640),
    pose=_RTMPOSE_L_URL,
    pose_input_size=(288, 384),
    backend="onnxruntime",
    device="cpu",
)


def _preprocess(frame: np.ndarray) -> np.ndarray:
    """CLAHE + unsharp mask to lift contrast and counter motion blur."""
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(l)
    enhanced = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
    blur = cv2.GaussianBlur(enhanced, (0, 0), sigmaX=2)
    return cv2.addWeighted(enhanced, 1.5, blur, -0.5, 0)


class _OneEuroFilter:
    """One-Euro filter — http://cristal.univ-lille.fr/~casiez/1euro/

    Adaptive low-pass filter for noisy real-time signals. Two knobs:
      * min_cutoff (Hz):  lower = more smoothing when the signal is stable.
                         Try 0.5–2.0. Smaller = less jitter at rest.
      * beta:            how aggressively the cutoff rises with velocity.
                         Larger = more responsive to fast motion, less lag.
                         Try 0.005–0.05.
    Velocity itself is also filtered (d_cutoff) so a single noisy frame can
    not produce a velocity spike that propagates into the position estimate.
    Operates element-wise on numpy arrays.
    """

    def __init__(self, freq: float = 30.0, min_cutoff: float = 1.0,
                 beta: float = 0.02, d_cutoff: float = 1.0):
        self.freq = freq
        self.min_cutoff = min_cutoff
        self.beta = beta
        self.d_cutoff = d_cutoff
        self.x_prev: np.ndarray | None = None
        self.dx_prev: np.ndarray | None = None

    def _alpha(self, cutoff):
        tau = 1.0 / (2 * np.pi * cutoff)
        te = 1.0 / self.freq
        return 1.0 / (1.0 + tau / te)

    def __call__(self, x: np.ndarray, update_mask: np.ndarray | None = None) -> np.ndarray:
        """x: (N,) or (N, D) signal. update_mask: bool array of shape (N,) — only
        update positions where True; rows where False keep their previous state."""
        if self.x_prev is None:
            self.x_prev = x.copy()
            self.dx_prev = np.zeros_like(x)
            return x.copy()

        dx = (x - self.x_prev) * self.freq
        a_d = self._alpha(self.d_cutoff)
        dx_hat = a_d * dx + (1 - a_d) * self.dx_prev

        cutoff = self.min_cutoff + self.beta * np.abs(dx_hat)
        a = self._alpha(cutoff)
        x_hat = a * x + (1 - a) * self.x_prev

        if update_mask is not None:
            if x_hat.ndim == 2 and update_mask.ndim == 1:
                update_mask_b = update_mask[:, None]
            else:
                update_mask_b = update_mask
            x_hat = np.where(update_mask_b, x_hat, self.x_prev)
            dx_hat = np.where(update_mask_b, dx_hat, self.dx_prev)

        self.x_prev = x_hat
        self.dx_prev = dx_hat
        return x_hat.copy()


def _estimate_body_center(kp_xy: np.ndarray, kp_conf: np.ndarray, thr: np.ndarray) -> tuple[np.ndarray | None, int]:
    """Torso center from shoulders/hips; hands above head must not move this."""
    anchors = np.array([5, 6, 11, 12])
    ok = kp_conf[anchors] > thr[anchors]
    if ok.sum() < 2:
        return None, int(ok.sum())
    return kp_xy[anchors][ok].mean(axis=0), int(ok.sum())


def _estimate_body_scale(kp_xy: np.ndarray, kp_conf: np.ndarray, thr: np.ndarray) -> float:
    """Robust body scale in pixels, used to reject impossible one-frame jumps."""
    pairs = [(5, 6), (11, 12), (5, 11), (6, 12)]
    lengths = []
    for a, b in pairs:
        if kp_conf[a] > thr[a] and kp_conf[b] > thr[b]:
            lengths.append(np.linalg.norm(kp_xy[a] - kp_xy[b]))
    if lengths:
        return float(np.median(lengths))
    return 0.0


def _kp_spread(kp_xy: np.ndarray, kp_conf: np.ndarray, thr: np.ndarray) -> float:
    """Diagonal of the bounding box of confident keypoints.

    When RTMPose collapses (often when the recovery arm crosses above the head
    and the YOLOX crop becomes unusual), all keypoints cluster near a single
    point and the spread shrinks well below the body scale. We use this as an
    extra signal to reject whole-frame collapses that the torso-scale ratio
    test alone might miss.
    """
    high = kp_conf > thr
    if int(high.sum()) < 4:
        return 0.0
    pts = kp_xy[high]
    return float(np.linalg.norm(pts.max(axis=0) - pts.min(axis=0)))


# Torso "skeleton bones" used for the per-bone collapse detector. Each tuple
# names a pair of keypoint indices whose distance should stay roughly constant
# between consecutive frames (modulo body roll). A sudden shrink in *any* of
# them is a strong signal that the corresponding region of the pose has
# collapsed onto a single point — which the median-of-bones torso scale and
# the global keypoint-spread test both miss when the rest of the body is
# still detected at a sensible location.
_TORSO_BONES = ((5, 6), (11, 12), (5, 11), (6, 12))

# Limb segments used for shape-consistency gating. For crawl, one-frame
# misdetections often put a wrist/ankle into the torso region; that creates
# implausible limb lengths while torso still looks valid. Tracking an EWMA of
# these lengths helps reject those outlier updates.
_LIMB_BONES = (
    (5, 7), (7, 9), (6, 8), (8, 10),      # left/right upper+lower arm
    (11, 13), (13, 15), (12, 14), (14, 16) # left/right upper+lower leg
)


def _torso_bone_lengths(kp_xy: np.ndarray, kp_conf: np.ndarray, thr: np.ndarray) -> np.ndarray:
    """Per-bone torso lengths (NaN where either endpoint is below threshold)."""
    out = np.full(len(_TORSO_BONES), np.nan, dtype=np.float64)
    for i, (a, b) in enumerate(_TORSO_BONES):
        if kp_conf[a] > thr[a] and kp_conf[b] > thr[b]:
            out[i] = float(np.linalg.norm(kp_xy[a] - kp_xy[b]))
    return out


def _limb_bone_lengths(kp_xy: np.ndarray, kp_conf: np.ndarray, thr: np.ndarray) -> np.ndarray:
    """Per-bone limb lengths (NaN where either endpoint is below threshold)."""
    out = np.full(len(_LIMB_BONES), np.nan, dtype=np.float64)
    for i, (a, b) in enumerate(_LIMB_BONES):
        if kp_conf[a] > thr[a] and kp_conf[b] > thr[b]:
            out[i] = float(np.linalg.norm(kp_xy[a] - kp_xy[b]))
    return out


def _detect_swim_cap(
    frame: np.ndarray,
    prev_cap: np.ndarray | None = None,
    prev_r: float = 0.0,
    darkness_thr: int = 60,
    min_area: int = 800,
    max_area: int = 8000,
    max_aspect_ratio: float = 2.8,
    continuity_weight: float = 5.0,
    max_jump_px: float = 80.0,
    radius_change_factor: float = 0.5,
) -> tuple[np.ndarray | None, float]:
    """Locate the dark swim-cap blob; return (centroid_xy, radius_px).

    The cap is by far the most visually stable anchor for the swimmer's head
    in overhead pool footage — pool water is bright, skin is light, the cap
    is uniformly dark. We threshold dark pixels, take connected components
    of plausible cap-like size and aspect, and pick the best candidate. When
    `prev_cap` is given we hard-reject candidates that jumped further than
    `max_jump_px` from the previous cap or shrank/grew by more than
    `radius_change_factor` vs `prev_r` — both filter out lane reflections /
    shadows / partial cap occlusion that briefly produce tiny dark blobs
    far from the actual head.

    Returns (None, 0.0) if nothing qualifies.
    """
    if frame is None:
        return None, 0.0
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, darkness_thr, 255, cv2.THRESH_BINARY_INV)
    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kern)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kern)

    n_labels, _, stats, cents = cv2.connectedComponentsWithStats(mask)
    best_xy: np.ndarray | None = None
    best_r = 0.0
    best_score = -np.inf
    for i in range(1, n_labels):  # 0 is background
        area = int(stats[i, cv2.CC_STAT_AREA])
        if area < min_area or area > max_area:
            continue
        w = int(stats[i, cv2.CC_STAT_WIDTH])
        h = int(stats[i, cv2.CC_STAT_HEIGHT])
        ar = max(w, h) / max(min(w, h), 1)
        if ar > max_aspect_ratio:
            continue
        cx, cy = float(cents[i, 0]), float(cents[i, 1])
        r = float(np.sqrt(area / np.pi))
        if prev_cap is not None:
            d = float(np.linalg.norm(np.array([cx, cy]) - prev_cap))
            if d > max_jump_px:
                continue
            if prev_r > 1.0 and abs(r - prev_r) > prev_r * radius_change_factor:
                continue
            score = area - continuity_weight * d
        else:
            score = float(area)
        if score > best_score:
            best_score = score
            best_xy = np.array([cx, cy], dtype=np.float64)
            best_r = r
    return best_xy, best_r


def _select_best_person(keypoints: np.ndarray, scores: np.ndarray,
                        prev_center: np.ndarray | None,
                        prev_scale: float | None) -> int:
    """Pick the detection that best matches the tracked body.

    Without a previous frame, picks the highest mean-score detection. With one,
    blends mean confidence with proximity of the candidate's torso center to
    the previously tracked center (in body-scale units) so a phantom detection
    on splash/reflection cannot steal the track from the swimmer.
    """
    n = int(len(keypoints))
    if n == 0:
        return 0
    if n == 1 or prev_center is None or prev_scale is None or prev_scale <= 1e-3:
        return int(np.argmax(scores.mean(axis=1)))

    best_idx = int(np.argmax(scores.mean(axis=1)))
    best_score = -np.inf
    for i in range(n):
        c, _ = _estimate_body_center(keypoints[i], scores[i], _UPDATE_THR)
        mean_s = float(scores[i].mean())
        if c is None:
            score = mean_s - 1.0
        else:
            dist = float(np.linalg.norm(c - prev_center)) / prev_scale
            score = mean_s - 0.3 * dist
        if score > best_score:
            best_score = score
            best_idx = i
    return best_idx


def _infer_pose_with_flip(
    frame: np.ndarray,
    prev_center: np.ndarray | None,
    prev_scale: float | None,
) -> tuple[np.ndarray | None, np.ndarray | None]:
    """Run RTMPose on the frame *and* its 180°-rotated copy, keep the better one.

    On overhead crawl, RTMPose-L often misorients the swimmer: the extended
    forward arm looks just as "leg-like" as the actual legs, so the head
    sometimes ends up on the swim shorts and the ankles on the splashing hand.
    Empirically, the pass with the higher best-detection mean confidence is
    the correctly-oriented one. We pay ~2x inference cost and pick that pass.
    Coordinates from the flipped pass are mapped back to original-frame pixels.

    The "best person" inside each pass is chosen by `_select_best_person`,
    which already biases toward detections near the previously tracked body,
    so a phantom person on splash/reflection in either pass cannot steal the
    track.

    Returns (kp_xy, kp_conf) of shape (17, 2) / (17,) for the single best
    person in original-frame pixel coordinates, or (None, None) if neither
    pass detected anyone.
    """
    h, w = frame.shape[:2]

    def _eval(kp_arr, sc_arr):
        if kp_arr is None or len(kp_arr) == 0:
            return None, None, -np.inf
        idx = _select_best_person(kp_arr, sc_arr, prev_center, prev_scale)
        return kp_arr[idx], sc_arr[idx], float(sc_arr[idx].mean())

    kp_n, sc_n = _model(frame)
    best_n_xy, best_n_sc, score_n = _eval(kp_n, sc_n)

    kp_f, sc_f = _model(cv2.flip(frame, -1))
    if kp_f is not None and len(kp_f) > 0:
        kp_f = kp_f.copy()
        kp_f[..., 0] = w - kp_f[..., 0]
        kp_f[..., 1] = h - kp_f[..., 1]
    best_f_xy, best_f_sc, score_f = _eval(kp_f, sc_f)

    if not np.isfinite(score_n) and not np.isfinite(score_f):
        return None, None
    if score_f > score_n:
        return best_f_xy, best_f_sc
    return best_n_xy, best_n_sc


class _KPSmoother:
    """One-Euro filter on each (x, y) coord with last-known position fallback.

    Visible joints get One-Euro filtered; invisible joints keep their last
    smoothed position, with confidence decaying so the skeleton fades gracefully
    if a joint stays missing for many frames. A per-joint update threshold and
    jump gate are important for crawl: fast, blurred feet often produce plausible
    but wrong ankle points for one or two frames.
    """

    def __init__(self, fps: float = 30.0, min_cutoff: float = 1.0, beta: float = 0.02,
                 update_thr: np.ndarray | float = _UPDATE_THR, conf_decay: float = 0.85,
                 max_jump_per_joint: np.ndarray = _MAX_JUMP_BY_JOINT,
                 max_body_jump_scale: float = 1.10,
                 scale_ratio_range: tuple[float, float] = (0.45, 2.20),
                 min_spread_ratio: float = 1.35,
                 min_bone_ratio: float = 0.45,
                 limb_ratio_range: tuple[float, float] = (0.55, 1.80),
                 strict_conf_for_long_jump: float = 0.65,
                 min_trusted_joints: int = 6,
                 velocity_momentum: float = 0.75,
                 prediction_decay: float = 0.90,
                 hold_conf_frames: int = 10,
                 hold_conf_decay: float = 0.97,
                 max_missed_frames: int = 18,
                 bone_ewma_alpha: float = 0.30,
                 limb_ewma_alpha: float = 0.22,
                 cap_max_distance_scale: float = 1.0,
                 cap_min_scale_from_radius: float = 4.0):
        self.filter = _OneEuroFilter(freq=fps, min_cutoff=min_cutoff, beta=beta)
        update_thr = np.asarray(update_thr, dtype=np.float64)
        if update_thr.ndim == 0:
            update_thr = np.full(17, float(update_thr))
        self.update_thr = update_thr
        self.conf_decay = conf_decay
        self.max_jump_per_joint = np.asarray(max_jump_per_joint, dtype=np.float64)
        self.max_body_jump_scale = max_body_jump_scale
        self.scale_ratio_range = scale_ratio_range
        self.min_spread_ratio = min_spread_ratio
        self.min_bone_ratio = min_bone_ratio
        self.limb_ratio_range = limb_ratio_range
        self.strict_conf_for_long_jump = strict_conf_for_long_jump
        self.min_trusted_joints = min_trusted_joints
        self.velocity_momentum = velocity_momentum
        self.prediction_decay = prediction_decay
        self.hold_conf_frames = hold_conf_frames
        self.hold_conf_decay = hold_conf_decay
        self.max_missed_frames = max_missed_frames
        self.bone_ewma_alpha = bone_ewma_alpha
        self.limb_ewma_alpha = limb_ewma_alpha
        self.cap_max_distance_scale = cap_max_distance_scale
        self.cap_min_scale_from_radius = cap_min_scale_from_radius
        self._conf: np.ndarray | None = None    # (17,)
        self._xy: np.ndarray | None = None      # (17, 2)
        self._vel: np.ndarray | None = None     # (17, 2) px/s
        self._center: np.ndarray | None = None  # (2,)
        self._scale: float | None = None
        # Per-bone EWMA of recent good torso lengths, used by the collapse
        # detector. Initialised from the first good frame.
        self._bone_ewma: np.ndarray | None = None  # (4,)
        # Per-bone EWMA of limb lengths; catches impossible arm/leg geometry.
        self._limb_ewma: np.ndarray | None = None  # (8,)
        # Number of consecutive frames each joint has been "held" without a
        # real measurement. Used as the "lost" signal so the visible
        # confidence does not have to decay just because we held the pose.
        self._missed_frames: np.ndarray = np.zeros(17, dtype=np.int32)
        # Swim-cap anchor: the dark cap is the most stable visual feature
        # in overhead pool footage. When present, we use it to validate the
        # model's head placement (and feed `_cap_xy` back to the detector
        # as a continuity prior on the next frame). `_pose_anchor_cap` is
        # the cap position at the moment the current `_xy` was last
        # established — frame-to-frame cap displacement is then used to
        # rigid-translate the held pose during long streaks of rejected
        # frames, so the skeleton tracks the swimmer instead of freezing.
        self._cap_xy: np.ndarray | None = None
        self._cap_r: float = 0.0
        self._pose_anchor_cap: np.ndarray | None = None

    def _predict_step(self, translate_to_cap: np.ndarray | None = None
                      ) -> tuple[np.ndarray | None, np.ndarray | None]:
        """Advance one frame without trusting new detections.

        If `translate_to_cap` is given and we have a previous anchor cap,
        rigid-translate the whole skeleton by the cap displacement so the
        held pose tracks the swimmer's actual head position. Otherwise fall
        back to velocity-based forward prediction (which usually decays to
        a frozen pose after a couple of frames).
        """
        if self._xy is None or self._conf is None:
            return None, None
        if self._vel is None:
            self._vel = np.zeros_like(self._xy)

        if translate_to_cap is not None and self._pose_anchor_cap is not None:
            delta = translate_to_cap - self._pose_anchor_cap
            self._xy = self._xy + delta
            if self._center is not None:
                self._center = self._center + delta
            self._pose_anchor_cap = translate_to_cap.copy()
            # Velocity is meaningless once we're being dragged by the cap.
            self._vel *= self.prediction_decay
        else:
            self._xy = self._xy + self._vel / max(self.filter.freq, 1e-6)
            self._vel *= self.prediction_decay

        next_miss = np.minimum(
            self._missed_frames + 1, np.iinfo(self._missed_frames.dtype).max
        )
        grace = next_miss <= self.hold_conf_frames
        floor = np.minimum(0.98, self.update_thr + 0.03)
        self._conf = np.where(grace, np.maximum(self._conf, floor), self._conf * self.hold_conf_decay)
        self._missed_frames = next_miss.astype(np.int32)
        return self._xy.copy(), self._conf.copy()

    def predict_only(self, current_cap: np.ndarray | None = None
                     ) -> tuple[np.ndarray | None, np.ndarray | None]:
        """Public fallback for frames where detector returns no person.

        When a cap detection is available we use it to translate the held
        pose (see `_predict_step`).
        """
        return self._predict_step(translate_to_cap=current_cap)

    def update(self, kp_xy: np.ndarray, kp_conf: np.ndarray,
               cap_xy: np.ndarray | None = None,
               cap_r: float = 0.0) -> tuple[np.ndarray, np.ndarray]:
        kp_xy = kp_xy.astype(np.float64)
        kp_conf = kp_conf.astype(np.float64)
        update = kp_conf > self.update_thr  # (17,) bool
        center, n_anchors = _estimate_body_center(kp_xy, kp_conf, self.update_thr)
        scale = _estimate_body_scale(kp_xy, kp_conf, self.update_thr)
        spread = _kp_spread(kp_xy, kp_conf, self.update_thr)
        bones = _torso_bone_lengths(kp_xy, kp_conf, self.update_thr)
        limb_bones = _limb_bone_lengths(kp_xy, kp_conf, self.update_thr)

        # If the torso suddenly jumps or changes size, the detector probably made
        # a bad crop/pose when the recovery arm passed above the head. Hold the
        # previous pose for this frame instead of moving every point.
        reject_pose = False
        if self._center is not None and self._scale is not None:
            if center is None or n_anchors < 2 or scale <= 1e-3:
                reject_pose = True
            else:
                center_jump = np.linalg.norm(center - self._center)
                scale_ratio = scale / max(self._scale, 1e-3)
                lo, hi = self.scale_ratio_range
                reject_pose = (
                    center_jump > self.max_body_jump_scale * self._scale
                    or scale_ratio < lo
                    or scale_ratio > hi
                )
            # Whole-skeleton collapse: every confident keypoint cluster onto a
            # tiny region. Common when the YOLOX crop is wrong while the
            # swimmer's arm is above the head.
            if not reject_pose and spread > 0 and spread < self.min_spread_ratio * self._scale:
                reject_pose = True
            # Per-bone collapse: the *longest* currently-visible torso bone
            # is well below its recent EWMA. Using max-of-ratios (not any)
            # distinguishes a real collapse — where every bone shrinks
            # together — from body roll, where shoulder-shoulder (and a
            # little hip-hip) foreshorten dramatically but the shoulder-hip
            # diagonals stay near full length. Body roll happens every
            # breath cycle in crawl and must not be flagged as a collapse,
            # otherwise the EWMA never updates and the skeleton freezes.
            if not reject_pose and self._bone_ewma is not None:
                with np.errstate(invalid="ignore", divide="ignore"):
                    ratios = bones / self._bone_ewma
                ok_b = (
                    np.isfinite(ratios)
                    & np.isfinite(self._bone_ewma)
                    & (self._bone_ewma > 1e-3)
                )
                if ok_b.any() and float(ratios[ok_b].max()) < self.min_bone_ratio:
                    reject_pose = True

        # If too few joints survived gating, this frame is not reliable enough
        # to update the track state.
        trusted_joints = int(update.sum())
        if trusted_joints < self.min_trusted_joints:
            reject_pose = True

        # If we've been holding for many frames, the held pose is stale: the
        # swimmer has almost certainly moved past it. Allow the new detection
        # through so the per-joint jump gate (with "lost" joints bypassed)
        # can re-anchor the skeleton instead of staying frozen forever.
        # (Cap-anchor checks below explicitly re-reject when the cap evidence
        # contradicts the model, so this override only relaxes *temporal*
        # rejections — not external evidence.)
        if reject_pose and int(self._missed_frames.min()) >= self.max_missed_frames:
            reject_pose = False

        # Swim-cap anchor (runs LAST so it has final say, overriding the
        # max-missed-frames re-acceptance above): the cap is our one
        # external piece of evidence about where the actual head is, and
        # the temporal gates alone are known to let RTMPose's inverted
        # "head-on-shorts" failure mode through frame after frame.
        #
        # If we've previously seen a cap (i.e. cap-anchor is in use for
        # this video) but the cap is missing on *this* frame, we have no
        # way to validate orientation — reject and predict instead.
        if cap_xy is None and self._pose_anchor_cap is not None:
            reject_pose = True

        if cap_xy is not None:
            head_idx = np.array([0, 1, 2, 3, 4])
            head_ok = kp_conf[head_idx] > self.update_thr[head_idx]
            if head_ok.any():
                head_xy = kp_xy[head_idx][head_ok].mean(axis=0)
                ref_scale = max(
                    scale,
                    self._scale if self._scale is not None else 0.0,
                    float(cap_r) * self.cap_min_scale_from_radius,
                )
                if (ref_scale > 1e-3 and
                        float(np.linalg.norm(head_xy - cap_xy))
                        > self.cap_max_distance_scale * ref_scale):
                    reject_pose = True

            # Structural orientation: the model's HIPS should sit much
            # farther from the cap than the model's HEAD does. In the
            # inverted-body failure mode RTMPose places its "hips" right
            # on top of the actual head (i.e. on the cap) and its "head"
            # down on the actual hip / shorts region — so the ratio
            # d(head, cap) / d(hip, cap) flips from << 1 to >> 1.
            if head_ok.any():
                hip_idx = np.array([11, 12])
                hip_ok = kp_conf[hip_idx] > self.update_thr[hip_idx]
                if hip_ok.any():
                    hip_xy = kp_xy[hip_idx][hip_ok].mean(axis=0)
                    d_head_cap = float(np.linalg.norm(head_xy - cap_xy))
                    d_hip_cap = float(np.linalg.norm(hip_xy - cap_xy))
                    if d_hip_cap < max(d_head_cap, float(cap_r)) * 1.2:
                        reject_pose = True

        if reject_pose:
            if self._xy is not None and self._conf is not None:
                # Predict through bad frames and keep confidence for a short
                # grace window so keypoints do not instantly disappear on
                # occlusion. When a cap is detected, drag the held pose
                # along with it so the skeleton tracks the swimmer.
                xy_p, conf_p = self._predict_step(translate_to_cap=cap_xy)
                return xy_p, conf_p
            # No prior state and the first measurement is itself rejected —
            # do NOT seed the smoother with a known-bad pose. Stay empty;
            # we'll initialize from the first measurement that passes the
            # gates.
            return None, None

        if not reject_pose:
            # Per-joint temporal gating, measured in *body-relative* pixels:
            # we subtract the body center's translation between frames before
            # checking how far each joint moved. This way pure locomotion
            # (swimmer translating through the frame with a fixed camera, or
            # the camera lagging behind a moving swimmer) contributes nothing
            # to the jump — only joint motion *relative to the rest of the
            # body* counts, which is what we actually want to gate. Without
            # this correction, a tight cap on head/shoulders/hips rejects
            # every new measurement during translation, confidence decays
            # frame by frame, and those joints fade out of the overlay.
            #
            # Joints that have been held for `max_missed_frames` are treated
            # as "lost" and bypass the gate so the skeleton can re-acquire a
            # joint that reappears after a long bad streak.
            #
            # We also accept long jumps when the model produces a strong new
            # detection (conf >= strict_conf_for_long_jump) AND the held
            # joint's confidence has decayed well below the new evidence
            # (held < 0.4 * new). That covers fast-recovery strokes where a
            # wrist/elbow legitimately moves > body_scale in one frame
            # while the per-joint cap would otherwise freeze it for the
            # rest of the clip (e.g. frame 32 R_elbow: model gives 0.81 at
            # the visible recovery arm, held position decayed to 0.12, but
            # the 167px jump exceeds the 112px cap).
            if self._xy is not None and scale > 1e-3:
                if (self._center is not None and center is not None):
                    body_shift = center - self._center
                    rel_jump = np.linalg.norm(
                        kp_xy - (self._xy + body_shift), axis=1
                    )
                else:
                    rel_jump = np.linalg.norm(kp_xy - self._xy, axis=1)
                lost = self._missed_frames > self.max_missed_frames
                strong_new = kp_conf >= self.strict_conf_for_long_jump
                if self._conf is not None:
                    strong_new &= self._conf < (kp_conf * 0.4)
                update &= (
                    lost
                    | strong_new
                    | (rel_jump < self.max_jump_per_joint * scale)
                )

            # Limb-shape guard: if a limb segment suddenly becomes implausible
            # relative to recent accepted geometry, only trust it when model
            # confidence is very high. This blocks one-frame wrist/ankle flips
            # during splash/occlusion without freezing normal motion.
            if self._limb_ewma is not None:
                lo, hi = self.limb_ratio_range
                bad_joint = np.zeros(17, dtype=bool)
                for idx, (a, b) in enumerate(_LIMB_BONES):
                    cur = limb_bones[idx]
                    ref = self._limb_ewma[idx]
                    if not np.isfinite(cur) or not np.isfinite(ref) or ref <= 1e-3:
                        continue
                    ratio = cur / ref
                    if ratio < lo or ratio > hi:
                        suspect = a if kp_conf[a] <= kp_conf[b] else b
                        bad_joint[suspect] = True
                update &= ~(bad_joint & (kp_conf < self.strict_conf_for_long_jump))

        prev_xy = self._xy.copy() if self._xy is not None else None
        smoothed_xy = self.filter(kp_xy, update_mask=update)

        if self._conf is None:
            self._conf = np.where(update, kp_conf, 0.0)
        else:
            new_conf = self._conf.copy()
            new_conf[update] = kp_conf[update]
            new_conf[~update] *= self.conf_decay
            self._conf = new_conf

        # Update joint velocities from accepted measurements only.
        if self._vel is None:
            self._vel = np.zeros_like(smoothed_xy)
        if prev_xy is not None:
            dt = 1.0 / max(self.filter.freq, 1e-6)
            raw_vel = (smoothed_xy - prev_xy) / dt
            m = self.velocity_momentum
            up2 = update[:, None]
            self._vel = np.where(up2, m * self._vel + (1 - m) * raw_vel, self._vel * self.prediction_decay)

        # A joint that just got a real measurement resets its missed counter;
        # otherwise the streak grows by one.
        self._missed_frames = np.where(update, 0, self._missed_frames + 1).astype(np.int32)

        self._xy = smoothed_xy.copy()
        if cap_xy is not None:
            # Remember the cap position at the moment this pose was
            # established. `_predict_step` uses cap displacement vs this
            # anchor to drag the held pose along with the swimmer through
            # rejected-frame streaks.
            self._pose_anchor_cap = cap_xy.copy()
        if not reject_pose and center is not None and scale > 1e-3:
            self._center = center.copy()
            self._scale = scale
            # EWMA of torso bone lengths over accepted frames only.
            valid_b = np.isfinite(bones)
            if self._bone_ewma is None:
                self._bone_ewma = np.where(valid_b, bones, np.nan)
            else:
                a = self.bone_ewma_alpha
                prev = self._bone_ewma
                new = np.where(
                    valid_b & np.isfinite(prev),
                    a * np.nan_to_num(bones) + (1 - a) * np.nan_to_num(prev),
                    np.where(valid_b, bones, prev),
                )
                self._bone_ewma = new

            # EWMA of limb lengths over accepted frames only.
            valid_l = np.isfinite(limb_bones)
            if self._limb_ewma is None:
                self._limb_ewma = np.where(valid_l, limb_bones, np.nan)
            else:
                a_l = self.limb_ewma_alpha
                prev_l = self._limb_ewma
                new_l = np.where(
                    valid_l & np.isfinite(prev_l),
                    a_l * np.nan_to_num(limb_bones) + (1 - a_l) * np.nan_to_num(prev_l),
                    np.where(valid_l, limb_bones, prev_l),
                )
                self._limb_ewma = new_l
        return smoothed_xy, self._conf


def _draw_person(frame, kp_xy, vis):
    for a, b in COCO_CONNECTIONS:
        if vis[a] and vis[b]:
            cv2.line(
                frame,
                (int(kp_xy[a, 0]), int(kp_xy[a, 1])),
                (int(kp_xy[b, 0]), int(kp_xy[b, 1])),
                (0, 200, 255), 2, cv2.LINE_AA,
            )
    for i, (x, y) in enumerate(kp_xy):
        if vis[i]:
            if i >= 11:
                color, r = (0, 80, 255), 5     # legs — red/orange
            elif i in (7, 8, 9, 10):
                color, r = (255, 120, 0), 4    # arms — blue
            else:
                color, r = (0, 255, 0), 4      # head/torso — green
            cv2.circle(frame, (int(x), int(y)), r, color, -1)


def annotate_video(
    video_path: str,
    output_path: str = "annotated.mp4",
    kp_conf_thr: np.ndarray = _DEFAULT_THR,
    update_conf_thr: np.ndarray = _UPDATE_THR,
    min_cutoff: float = 1.0,
    beta: float = 0.02,
    max_jump_per_joint: np.ndarray = _MAX_JUMP_BY_JOINT,
    max_body_jump: float = 1.10,
    limb_ratio_range: tuple[float, float] = (0.55, 1.80),
    strict_conf_for_long_jump: float = 0.65,
    min_trusted_joints: int = 6,
    velocity_momentum: float = 0.75,
    prediction_decay: float = 0.90,
    hold_conf_frames: int = 10,
    hold_conf_decay: float = 0.97,
    preprocess: bool = True,
    use_orientation_flip: bool = True,
    use_cap_anchor: bool = False,
    cap_max_distance_scale: float = 1.0,
) -> str:
    """Detect body keypoints on every frame with RTMPose-L and write annotated video.

    RTMPose-L (384x288 input, body7 training set) is significantly more accurate
    than YOLO-pose / MediaPipe on out-of-distribution poses like swimming. Uses
    YOLOX-M for person detection, then crops + runs RTMPose on the largest box.

    Smoothing uses the One-Euro filter — a velocity-adaptive low-pass that
    eliminates jitter at rest while staying responsive during fast motion.
    Two knobs:
      * min_cutoff (Hz) — lower = more smoothing when joint is stable (0.5..2)
      * beta            — higher = less lag during fast motion (0.005..0.05)

    Args:
        video_path:         Input video path.
        output_path:        Output annotated video path.
        kp_conf_thr:        Array (17,) of per-joint confidence thresholds for drawing.
        update_conf_thr:    Array (17,) of stricter thresholds for updating tracking state.
        min_cutoff:         One-Euro min cutoff frequency.
        beta:               One-Euro responsiveness coefficient.
        max_jump_per_joint: Array (17,) of per-joint one-frame jump caps as
                            multiples of body (torso) scale. Catches collapses
                            of individual limbs when the recovery arm crosses
                            above the head and RTMPose mislocalizes joints.
        max_body_jump:      Max allowed one-frame torso jump as a multiple of body scale.
        limb_ratio_range:   Allowed min/max ratio vs recent limb-length EWMA.
                            Narrower range rejects more single-frame flips.
        strict_conf_for_long_jump:
                            Limb outliers are accepted only if confidence is
                            above this value (0..1).
        min_trusted_joints: Minimum number of joints that must survive gating
                            before a frame can update tracker state.
        velocity_momentum:  Momentum for per-joint velocity estimate (0..1).
        prediction_decay:   Velocity decay used when pose is rejected.
        hold_conf_frames:   Keep keypoints visible this many missed frames.
        hold_conf_decay:    Confidence decay after grace window.
        preprocess:         Apply CLAHE + unsharp mask before inference.
        use_orientation_flip:
                            Run RTMPose on the frame and a 180°-rotated copy
                            and keep the higher-confidence detection. On
                            overhead crawl footage this empirically picks
                            the upright-pose interpretation more often than
                            a single pass on the raw orientation, giving an
                            articulating skeleton that follows the body
                            instead of locking onto an inverted prior.
                            On by default — most reliable improvement we
                            have on this footage.
        use_cap_anchor:     Detect the dark swim-cap blob each frame and
                            reject poses whose head joints are too far from
                            the cap (or whose hips are closer to the cap
                            than their head). Off by default because the
                            cap detector occasionally loses the blob on
                            splash/occlusion, and the rejection then locks
                            the skeleton to a frozen template translated by
                            the cap displacement. Useful as an extra guard
                            on footage where the cap is consistently
                            visible.
        cap_max_distance_scale:
                            Maximum allowed distance, in body-scale units,
                            between the cap centroid and the mean predicted
                            head-joint position. Smaller = stricter.

    Returns:
        output_path
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 20.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f"Cannot open VideoWriter for {output_path}")

    smoother = _KPSmoother(
        fps=fps,
        min_cutoff=min_cutoff,
        beta=beta,
        update_thr=update_conf_thr,
        max_jump_per_joint=max_jump_per_joint,
        max_body_jump_scale=max_body_jump,
        limb_ratio_range=limb_ratio_range,
        strict_conf_for_long_jump=strict_conf_for_long_jump,
        min_trusted_joints=min_trusted_joints,
        velocity_momentum=velocity_momentum,
        prediction_decay=prediction_decay,
        hold_conf_frames=hold_conf_frames,
        hold_conf_decay=hold_conf_decay,
        cap_max_distance_scale=cap_max_distance_scale,
    )

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        infer_frame = _preprocess(frame) if preprocess else frame

        # Swim-cap detection runs on the *raw* frame — preprocessing's CLAHE
        # lifts dark pixels and dilutes the contrast between cap and skin.
        # Seeding with the previous cap stops a transient dark blob from
        # stealing the anchor; passing `prev_r` also rejects sudden radius
        # changes that signal a wrong blob.
        if use_cap_anchor:
            cap_xy, cap_r = _detect_swim_cap(
                frame, prev_cap=smoother._cap_xy, prev_r=smoother._cap_r
            )
            if cap_xy is not None:
                smoother._cap_xy = cap_xy
                smoother._cap_r = cap_r
        else:
            cap_xy, cap_r = None, 0.0

        if use_orientation_flip:
            # Dual-orientation inference: pick whichever pass (normal or
            # 180°-flipped) produced a higher-confidence best detection.
            # Recovers correct head/feet assignment on overhead crawl where
            # RTMPose otherwise places legs onto the extended hand.
            best_xy, best_sc = _infer_pose_with_flip(
                infer_frame, smoother._center, smoother._scale
            )
        else:
            keypoints, scores = _model(infer_frame)
            if keypoints is not None and len(keypoints) > 0:
                # Prefer the detection closest to the previously tracked body
                # so a phantom box on splash/reflection can't steal the track.
                idx = _select_best_person(
                    keypoints, scores, smoother._center, smoother._scale
                )
                best_xy, best_sc = keypoints[idx], scores[idx]
            else:
                best_xy = best_sc = None

        if best_xy is not None:
            kp_xy_s, kp_conf_s = smoother.update(
                best_xy, best_sc, cap_xy=cap_xy, cap_r=cap_r
            )
        else:
            # No detection on this frame: keep track alive via prediction,
            # dragged by the cap when available.
            kp_xy_s, kp_conf_s = smoother.predict_only(current_cap=cap_xy)

        if kp_xy_s is not None and kp_conf_s is not None:
            vis = kp_conf_s > kp_conf_thr
            if vis.sum() >= 3:
                _draw_person(frame, kp_xy_s, vis)

        # Debug overlay: faint yellow ring around the detected cap so it's
        # obvious when the anchor is or isn't being found.
        if cap_xy is not None and cap_r > 0:
            cv2.circle(frame, (int(cap_xy[0]), int(cap_xy[1])),
                       int(max(cap_r, 12)), (0, 255, 255), 2, cv2.LINE_AA)

        writer.write(frame)
        frame_idx += 1

    cap.release()
    writer.release()
    print(f"Done — {frame_idx} frames written to {output_path}")
    return output_path

load /Users/alex/.cache/rtmlib/hub/checkpoints/yolox_m_8xb8-300e_humanart-c2c7a14a.onnx with onnxruntime backend
load /Users/alex/.cache/rtmlib/hub/checkpoints/rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.onnx with onnxruntime backend


In [ ]:
annotate_video(
    video_path="data/Брасс сверху.mp4",
    output_path="annotated.mp4",
    limb_ratio_range=(0.68, 1.45),
    strict_conf_for_long_jump=0.75,
    min_trusted_joints=7,
    beta=0.012,
    min_cutoff=1.3,
    prediction_decay=0.90,
    hold_conf_frames=14,
    hold_conf_decay=0.985,
)

Done — 65 frames written to annotated.mp4


'annotated.mp4'